## 1. Configuración Inicial y Carga de Datos

En esta primera sección importamos todas las herramientas necesarias (traductor, detector de idiomas y utilidades de paralelización). A continuación, cargamos el dataset df_comentarios_limpio.csv generado en el paso anterior y aplicamos una pequeña limpieza de valores nulos o vacíos para que no interfieran en el proceso de traducción.

In [ ]:
# Instalación de librerías (descomentar si es necesario)
# !pip install deep-translator langdetect -q

# Librerías
import pandas as pd
import os
import time
from langdetect import detect
from langdetect.detector_factory import LangDetectException
from deep_translator import GoogleTranslator
from concurrent.futures import ThreadPoolExecutor, as_completed

# Función para limpiar nulos o cadenas vacías antes de traducir
def gestionar_nulos(df, columnas):
    for col in columnas:
        df[col] = df[col].fillna('')
    # Nos quedamos con las filas que NO tengan todas las columnas de interés vacías
    df = df[~df[columnas].apply(lambda row: all(v == '' for v in row), axis=1)]
    return df

# Carga de datos
ruta_entrada = 'datos/procesados/df_comentarios_limpio.csv'
df_comentarios_limpio = pd.read_csv(ruta_entrada)

# Limpieza de nulos inicial
df_comentarios_limpio = gestionar_nulos(df_comentarios_limpio, df_comentarios_limpio.columns.tolist())
print(f"Dimensiones del dataset listo para traducir: {df_comentarios_limpio.shape}")
display(df_comentarios_limpio.head(3))

## 2. Motores de Detección y Traducción en Lote

Aquí definimos las dos funciones principales (los motores) de nuestra traducción. La primera (detectar_idioma) analiza cada texto para saber si ya está en español (y así ahorrar tiempo y peticiones). La segunda (traducir_lote) se encarga de enviar paquetes de textos a la API de Google Translate, incorporando un sistema de reintentos automáticos (con pausas) por si el servidor bloquea las peticiones temporalmente.

In [ ]:
def detectar_idioma(texto):
    """Detecta el idioma de un texto. Si está vacío o falla, asume 'es' o 'desconocido'."""
    try:
        if pd.isna(texto) or str(texto).strip() == '':
            return 'es'
        return detect(str(texto))
    except LangDetectException:
        return 'desconocido'

def traducir_lote(textos):
    """Traduce una lista de textos al español con sistema de reintentos (rate limit bypass)."""
    intentos = 3
    for intento in range(intentos):
        try:
            return GoogleTranslator(source='auto', target='es').translate_batch(textos)
        except Exception as e:
            print(f"Error en lote (intento {intento+1}/{intentos}): {e}")
            if intento < intentos - 1:
                espera = 60 * (intento + 1)  # Espera progresiva: 60s, luego 120s
                print(f"  Esperando {espera}s antes de reintentar...")
                time.sleep(espera)
    return textos  # Si fallan todos los intentos, devuelve los originales para no perder datos

## 3. Orquestador de Traducción con Checkpoints

Esta es la función principal que orquesta el trabajo. Realiza tres tareas clave:

1. Filtra solo los textos que NO están en español.

2. Comprueba si existe un "checkpoint" previo (por si el proceso se interrumpió y queremos retomarlo por donde iba).

3. Envía los textos a traducir usando hilos paralelos (ThreadPoolExecutor) para acelerar el proceso, guardando el progreso cada cierto número de lotes.


In [ ]:
def traduccion(columna, tamaño_lote=50, max_workers=2, checkpoint_path='checkpoint.csv', delay_between_batches=3):
    print("  Detectando idiomas...")
    idiomas = columna.apply(detectar_idioma)

    mask_traducir = idiomas != 'es'
    indices_traducir = columna[mask_traducir].index
    textos_traducir = columna[mask_traducir].tolist()

    print(f"  Total: {len(columna)} | En español: {(~mask_traducir).sum()} | A traducir: {mask_traducir.sum()}")

    resultado = columna.copy()

    # Cargar checkpoint si existe (por si el proceso se cayó antes)
    if os.path.exists(checkpoint_path):
        print("  Cargando checkpoint previo...")
        checkpoint = pd.read_csv(checkpoint_path, index_col=0).squeeze()
        ya_hechos = checkpoint.index
        indices_traducir = indices_traducir.difference(ya_hechos)
        textos_traducir = columna[indices_traducir].tolist()
        resultado.update(checkpoint)
        print(f"  Ya traducidos en ejecuciones previas: {len(ya_hechos)} | Quedan: {len(textos_traducir)}")

    # Crear lotes
    lotes = [
        (textos_traducir[i:i+tamaño_lote], indices_traducir[i:i+tamaño_lote])
        for i in range(0, len(textos_traducir), tamaño_lote)
    ]
    total_lotes = len(lotes)
    completados = 0

    if total_lotes == 0:
        print("¡Nada que traducir!")
        return resultado

    # Traducir en paralelo
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futuros = {
            executor.submit(traducir_lote, lote): indices
            for lote, indices in lotes
        }

        for futuro in as_completed(futuros):
            indices_lote = futuros[futuro]
            try:
                traducidos = futuro.result()
                for idx, texto in zip(indices_lote, traducidos):
                    resultado[idx] = texto
            except Exception as e:
                print(f"  Error fatal en lote: {e}")

            completados += 1

            # Guardar checkpoint cada 50 lotes completados
            if completados % 50 == 0:
                resultado[mask_traducir].to_csv(checkpoint_path)
                print(f"  {completados}/{total_lotes} lotes completados ({completados/total_lotes*100:.1f}%) -- Checkpoint guardado")

            # Retardo para evitar bloqueos por límite de peticiones (rate limit)
            time.sleep(delay_between_batches)

    # Guardar checkpoint final y retornar
    resultado[mask_traducir].to_csv(checkpoint_path)
    print("¡Traducción de esta columna completada!")
    return resultado

## 4. Ejecución por Columnas

Aplicamos la función orquestadora sobre las tres columnas de interés: comentario general, aspectos positivos y aspectos negativos. Configuramos tiempos de espera y tamaños de lote conservadores para evitar que la API gratuita de Google nos bloquee por exceso de peticiones.

In [ ]:
# 1. Traducción del Comentario General
print("Iniciando traducción de: 'comentario_general'")
df_comentarios_limpio['comentario_general'] = traduccion(
    df_comentarios_limpio['comentario_general'],
    tamaño_lote=25,
    max_workers=1,
    checkpoint_path='datos/procesados/ckpt_comentario_general.csv',
    delay_between_batches=25 # Retardo alto para evitar el límite de peticiones
)

In [ ]:
# 2. Traducción de los Aspectos Positivos
print("\nIniciando traducción de: 'positivo'")
df_comentarios_limpio['positivo'] = traduccion(
    df_comentarios_limpio['positivo'],
    tamaño_lote=32,
    max_workers=1,
    checkpoint_path='datos/procesados/ckpt_positivo.csv',
    delay_between_batches=5 
)

In [ ]:
# 3. Traducción de los Aspectos Negativos
print("\nIniciando traducción de: 'negativo'")
df_comentarios_limpio['negativo'] = traduccion(
    df_comentarios_limpio['negativo'],
    tamaño_lote=32,
    max_workers=1,
    checkpoint_path='datos/procesados/ckpt_negativo.csv',
    delay_between_batches=5 
)

## 5. Exportación del Dataset Final Traducido

Finalmente, una vez que todas las columnas de texto han sido estandarizadas al español, guardamos el DataFrame resultante. Este dataset es el que finalmente usaremos para tareas analíticas avanzadas, como análisis de sentimiento o modelado de tópicos.

In [ ]:
ruta_final = 'datos/procesados/comentarios_traducidos_final.csv'
df_comentarios_limpio.to_csv(ruta_final, index=False, encoding='utf-8-sig')

print(f"¡Proceso finalizado! Archivo guardado con éxito en: {ruta_final}")
display(df_comentarios_limpio[['comentario_general', 'positivo', 'negativo']].head(5))